In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Fast and reproducible sample
df_full_info = pd.read_csv("../data/criteo-uplift-v2.1.csv", nrows=5)  # just to see columns
print(df_full_info.columns.tolist())

# Now load a proper random sample of 400,000 rows
df = pd.read_csv(
    "../data/criteo-uplift-v2.1.csv",
    skiprows=lambda x: x > 0 and x % 35 != 0,   # roughly every 35th row → ~400k rows
)

print("Working sample shape:", df.shape)
print(df["treatment"].value_counts(normalize=True).round(4))
print(df.groupby("treatment")["conversion"].mean())

KeyboardInterrupt: 

In [ ]:
# Naive ATE
conv_t = df.loc[df["treatment"] == 1, "conversion"]
conv_c = df.loc[df["treatment"] == 0, "conversion"]

ate_naive = conv_t.mean() - conv_c.mean()
print(f"Naive ATE: {ate_naive:.6f} ({ate_naive*100:.4f} percentage points)")

# Two-proportion z-test
count = np.array([conv_t.sum(), conv_c.sum()])
nobs  = np.array([len(conv_t), len(conv_c)])
z_stat, p_value = stats.ttest_ind(conv_t, conv_c, equal_var=False)  # or proportions_ztest
print(f"p-value (two-sided): {p_value:.2e}")

# Bootstrap CI (recommended)
def bootstrap_ate(df, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    ates = []
    for _ in range(n_boot):
        sample = df.sample(frac=1, replace=True, random_state=rng.integers(1e9))
        ate = (sample.loc[sample.treatment==1, "conversion"].mean() - 
               sample.loc[sample.treatment==0, "conversion"].mean())
        ates.append(ate)
    return np.percentile(ates, [2.5, 97.5])

ci_low, ci_high = bootstrap_ate(df, n_boot=500)  # 500 is enough for a first look
print(f"95% Bootstrap CI: [{ci_low:.6f}, {ci_high:.6f}]")

In [ ]:
features = [f"f{i}" for i in range(12)]

def standardized_mean_diff(df, feature, treatment_col="treatment"):
    t = df.loc[df[treatment_col]==1, feature]
    c = df.loc[df[treatment_col]==0, feature]
    return (t.mean() - c.mean()) / np.sqrt((t.var() + c.var()) / 2)

smds = {f: standardized_mean_diff(df, f) for f in features}
smd_series = pd.Series(smds).sort_values()

print("Standardized Mean Differences (SMD):")
print(smd_series.round(4))

# Rule of thumb: |SMD| < 0.1 is usually considered balanced
print("\nFeatures with |SMD| > 0.1:")
print(smd_series[smd_series.abs() > 0.1])